##**MINI PROJECT- NLP**

##**InterviewMate: An NLP-Based Chatbot for Practicing Interview Answers with Automated Feedback**


In [2]:
!pip install scikit-learn pandas nltk

In [3]:
# Install & Imports

import re
import random
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [4]:
FILLER_WORDS = ["um", "uh", "like", "basically", "actually", "you know", "sort of", "kind of"]

In [5]:
# SECTION 1: Load Dataset

def load_dataset(path="dataset/interview_qa.csv"):
    """Loads the interview Q&A dataset."""
    df = pd.read_csv(path)
    # keywords column is a semicolon-separated string -> convert to list
    df["keywords"] = df["keywords"].apply(lambda x: [k.strip().lower() for k in x.split(";")])
    return df


In [6]:
# SECTION 2: Preprocessing
# ============================================================
def preprocess(text):
    """
    Basic text preprocessing:
    - lowercase
    - remove punctuation/special characters
    - collapse extra whitespace
    Note: we deliberately do NOT remove stopwords here, because stopwords
    can matter for keeping the answer's TF-IDF vector meaningful when
    answers are short. This is discussed as a design choice in the report.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [7]:
# SECTION 3: Scoring Functions
# ============================================================
def compute_tfidf_similarity(model_answer, user_answer):
    """
    Computes cosine similarity between the model answer and the user's answer
    using TF-IDF. We fit the vectorizer on just these two documents so the
    comparison is self-contained per question (no need for a global corpus).
    """
    docs = [preprocess(model_answer), preprocess(user_answer)]
    if docs[1].strip() == "":
        return 0.0
    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(docs)
    except ValueError:
        # happens if user answer has no recognizable tokens after preprocessing
        return 0.0
    sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    return round(float(sim), 3)


def compute_keyword_coverage(user_answer, keywords):
    """
    Checks how many of the expected key concepts appear in the user's answer.
    Returns coverage ratio and list of missing keywords.
    """
    user_text = preprocess(user_answer)
    covered = []
    missing = []
    for kw in keywords:
        kw_clean = preprocess(kw)
        if kw_clean in user_text:
            covered.append(kw)
        else:
            missing.append(kw)
    coverage_ratio = len(covered) / len(keywords) if keywords else 0
    return round(coverage_ratio, 3), missing


def count_filler_words(user_answer):
    """Counts filler words as a simple proxy for confidence/fluency."""
    text = user_answer.lower()
    count = 0
    for filler in FILLER_WORDS:
        count += len(re.findall(r"\b" + re.escape(filler) + r"\b", text))
    return count


def compute_final_score(model_answer, user_answer, keywords):
    """
    Combines TF-IDF similarity (60%) and keyword coverage (40%) into a
    single score out of 100. Weights were chosen so semantic overlap
    (similarity) matters slightly more than exact keyword matching,
    since candidates often paraphrase concepts in their own words.
    """
    similarity = compute_tfidf_similarity(model_answer, user_answer)
    coverage, missing = compute_keyword_coverage(user_answer, keywords)
    score = round((0.6 * similarity + 0.4 * coverage) * 100, 1)
    fillers = count_filler_words(user_answer)
    return {
        "score": score,
        "similarity": similarity,
        "coverage": coverage,
        "missing_keywords": missing,
        "filler_count": fillers,
    }


In [8]:
# SECTION 4: Feedback Generator
# ============================================================
def generate_feedback(result):
    """Turns the numeric result into human-readable feedback."""
    score = result["score"]
    lines = []

    if score >= 75:
        lines.append("Strong answer! You covered most of the key points clearly.")
    elif score >= 50:
        lines.append("Decent answer, but there is room to add more depth or detail.")
    else:
        lines.append("This answer misses several important points. Consider revising it.")

    if result["missing_keywords"]:
        lines.append("Concepts you could add: " + ", ".join(result["missing_keywords"]) + ".")
    else:
        lines.append("You covered all the expected key concepts. Well done!")

    if result["filler_count"] > 0:
        lines.append(f"Note: detected {result['filler_count']} filler word(s) "
                      f"(e.g. 'um', 'like'). Try to reduce these for a more confident answer.")

    return "\n".join(lines)

In [15]:
# SECTION 5: Interactive Chatbot Loop  (run this to demo live)
# ============================================================
def run_chatbot(df, num_questions=5):
    """
    Simple CLI chatbot loop:
    - randomly picks questions
    - takes the user's typed answer
    - scores it and gives feedback
    - tracks the session average score
    """
    print("=" * 60)
    print(" Welcome to InterviewMate - your NLP interview practice bot")
    print("=" * 60)

    sample = df.sample(n=min(num_questions, len(df))).reset_index(drop=True)
    session_scores = []

    for i, row in sample.iterrows():
        print(f"\nQ{i+1} [{row['category']}]: {row['question']}")
        user_answer = input("Your answer: ")

        result = compute_final_score(row["model_answer"], user_answer, row["keywords"])
        session_scores.append(result["score"])

        print(f"\nScore: {result['score']}/100  "
              f"(similarity={result['similarity']}, keyword coverage={result['coverage']})")
        print(generate_feedback(result))

    if session_scores:
        avg = round(sum(session_scores) / len(session_scores), 1)
        print("\n" + "=" * 60)
        print(f" Session complete! Average score: {avg}/100")
        print("=" * 60)



In [22]:
# SECTION 6: Batch Evaluation (for Results/Evaluation section)
# ============================================================
# To evaluate how well the automated score aligns with human judgement,
# manually collect a small set of (question_id, sample_answer, human_label)
# where human_label is one of: "good", "average", "poor".
# Fill in real answers you tested with the chatbot, then run this section.

EVAL_SET = [
    (26, "An array stores elements in contiguous memory locations, so accessing an element by index is fast. A linked list stores elements as separate nodes connected using pointers, so insertion and deletion are easier, but accessing an element takes more time.", "good"),
    (16, "he main difference is that a list is mutable, which means we can add, remove, or modify elements after creating it. A tuple is immutable, so once it is created, its elements cannot be changed. Lists use square brackets, while tuples use parentheses ", "good"),
    (2, "I think i have the capability of learn faster any topics easily, so that i can easily adapt to your company culture and procedures also my new works there", "poor"),
    (31, "Cosine similarity is used to measure how similar two vectors or pieces of text are. It is commonly used in things like text similarity, document comparison, and recommendation systems.", "average"),
    (20, "A hash table is a data structure used to store data in key-value pairs. It uses a hash function to convert the key into an index,", "average"),

]


def run_batch_evaluation(df, eval_set):
    """
    Runs the scoring pipeline on a labeled evaluation set and prints
    a comparison table of automated score vs human label, for the
    report's Evaluation/Results section.
    """
    print(f"{'QID':<5}{'Score':<8}{'Bucket':<10}{'Human Label':<12}{'Match?':<6}")
    print("-" * 45)

    def score_to_bucket(score):
        if score >= 75:
            return "good"
        elif score >= 50:
            return "average"
        else:
            return "poor"

    matches = 0
    for qid, answer, label in eval_set:
        row = df[df["question_id"] == qid].iloc[0]
        result = compute_final_score(row["model_answer"], answer, row["keywords"])
        bucket = score_to_bucket(result["score"])
        match = "Yes" if bucket == label else "No"
        if match == "Yes":
            matches += 1
        print(f"{qid:<5}{result['score']:<8}{bucket:<10}{label:<12}{match:<6}")

    accuracy = round(matches / len(eval_set) * 100, 1) if eval_set else 0
    print("-" * 45)
    print(f"Agreement with human labels: {accuracy}% ({matches}/{len(eval_set)})")


In [23]:
# MAIN (for running as a plain script: python source_code.py)
# ============================================================
if __name__ == "__main__":
    dataset = load_dataset("interview_qa.csv")

    print("\n--- DEMO: single answer scoring ---")
    demo_row = dataset.iloc[15]  # "difference between list and tuple"
    demo_answer = "A list can change after you make it but a tuple stays the same."
    demo_result = compute_final_score(demo_row["model_answer"], demo_answer, demo_row["keywords"])
    print(demo_result)
    print(generate_feedback(demo_result))

    print("\n--- Starting interactive chatbot (type answers when prompted) ---")
    run_chatbot(dataset, num_questions=5)
    print("\n--- Batch evaluation against manually labeled answers ---")
    run_batch_evaluation(dataset, EVAL_SET)




--- DEMO: single answer scoring ---
{'score': 21.9, 'similarity': 0.098, 'coverage': 0.4, 'missing_keywords': ['mutable', 'immutable', 'brackets'], 'filler_count': 0}
This answer misses several important points. Consider revising it.
Concepts you could add: mutable, immutable, brackets.

--- Starting interactive chatbot (type answers when prompted) ---
 Welcome to InterviewMate - your NLP interview practice bot

Q1 [Technical]: What is the difference between an array and a linked list?
Your answer: An array stores elements in contiguous memory locations, so accessing an element by index is fast. A linked list stores elements as separate nodes connected using pointers, so insertion and deletion are easier, but accessing an element takes more time.

Score: 68.9/100  (similarity=0.481, keyword coverage=1.0)
Decent answer, but there is room to add more depth or detail.
You covered all the expected key concepts. Well done!

Q2 [Technical]: What is the difference between a list and a tuple 

In [26]:
EVAL_SET = [
    (26, "An array stores elements in contiguous memory locations, so accessing an element by index is fast. A linked list stores elements as separate nodes connected using pointers, so insertion and deletion are easier, but accessing an element takes more time.", "good"),
    (16, "he main difference is that a list is mutable, which means we can add, remove, or modify elements after creating it. A tuple is immutable, so once it is created, its elements cannot be changed. Lists use square brackets, while tuples use parentheses ", "good"),
    (2, "I think i have the capability of learn faster any topics easily, so that i can easily adapt to your company culture and procedures also my new works there", "poor"),
    (31, "Cosine similarity is used to measure how similar two vectors or pieces of text are. It is commonly used in things like text similarity, document comparison, and recommendation systems.", "average"),
    (20, "A hash table is a data structure used to store data in key-value pairs. It uses a hash function to convert the key into an index,", "average"),

]

run_batch_evaluation(dataset, EVAL_SET)

QID  Score   Bucket    Human Label Match?
---------------------------------------------
26   68.9    average   good        No    
16   75.8    good      good        Yes   
2    12.6    poor      poor        Yes   
31   46.2    poor      average     No    
20   60.6    average   average     Yes   
---------------------------------------------
Agreement with human labels: 60.0% (3/5)
